# 02 Exploratory Data Analysis

This notebook explores the datasets prepared in `01_load_data.ipynb`.

The goal is to understand data coverage, regional availability, missing values, indicator definitions, and candidate variables before building the AI Literacy Gap Index.  
At this stage, I separate exploratory analysis from final modelling decisions. The final index variables should only be selected after inspecting the available data.

- Step 1: Load each dataset separately
- Step 2: Inspect each dataset separately
- Step 3: Select candidate variables from each dataset
- Step 4: Filter each dataset to the chosen variable / year / geo level
- Step 5: Convert each dataset into one clean NUTS-1 feature table
- Step 6: Merge those feature tables into one final modelling dataframe

In [1]:
from pathlib import Path
import re
import json
import io
import urllib.request
import pandas as pd
import numpy as np

# -----------------------------
# Project paths
# -----------------------------
# This notebook is located in: CHL_DATA_SCIENCE_PROJECT/notebooks/
# The project root is therefore one level above the notebook folder.

CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name == "notebooks":
    PROJECT_DIR = CURRENT_DIR.parent
else:
    PROJECT_DIR = CURRENT_DIR

DATA_DIR = PROJECT_DIR / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
RESULTS_DIR = PROJECT_DIR / "results"
EDA_DIR = RESULTS_DIR / "eda"

for path in [RAW_DIR, PROCESSED_DIR, EDA_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print("Project directory:", PROJECT_DIR)
print("Processed data directory:", PROCESSED_DIR)
print("EDA results directory:", EDA_DIR)

# -----------------------------
# Load project data catalog
# -----------------------------
catalog_path = PROCESSED_DIR / "project_data_catalog.csv"

if not catalog_path.exists():
    raise FileNotFoundError(
        f"Data catalog not found at {catalog_path}. "
        "Please run 01_load_data.ipynb first."
    )

data_catalog = pd.read_csv(catalog_path)

#display(data_catalog)

Project directory: /Users/joshuakehrer/code/JoshK00/ai-literacy-gap-index/AI-Literacy-Gap-Index
Processed data directory: /Users/joshuakehrer/code/JoshK00/ai-literacy-gap-index/AI-Literacy-Gap-Index/data/processed
EDA results directory: /Users/joshuakehrer/code/JoshK00/ai-literacy-gap-index/AI-Literacy-Gap-Index/results/eda


## Load prepared datasets

The data loading notebook saved each dataset as a separate processed CSV file.  
In this step, I load all available processed datasets into a dictionary and create a compact overview of their shapes, time coverage, regional coverage, and missing values.

This gives a first check of which datasets are ready for EDA and which may need additional filtering or cleaning.

In [2]:
# -----------------------------
# Load all prepared datasets from the project catalog
# -----------------------------

datasets = {}
dataset_overview_rows = []

for _, row in data_catalog.iterrows():
    dataset_code = row["dataset_code"]
    processed_file = row["processed_file"]
    file_path = PROCESSED_DIR / processed_file

    if not file_path.exists():
        print(f"Skipping {dataset_code}: file not found at {file_path}")
        continue

    df = pd.read_csv(file_path)
    datasets[dataset_code] = df

    # Identify possible main value columns
    possible_value_cols = [
        "value",
        "population",
        "poverty_social_exclusion_rate"
    ]
    available_value_cols = [col for col in possible_value_cols if col in df.columns]
    main_value_col = available_value_cols[0] if available_value_cols else None

    overview = {
        "dataset_code": dataset_code,
        "dataset_name": row["dataset_name"],
        "pillar": row["pillar"],
        "n_rows": df.shape[0],
        "n_columns": df.shape[1],
        "main_value_col": main_value_col,
        "missing_share_main_value": df[main_value_col].isna().mean() if main_value_col else np.nan,
        "n_geo_codes": df["geo"].nunique() if "geo" in df.columns else np.nan,
        "min_year": df["year"].min() if "year" in df.columns else np.nan,
        "max_year": df["year"].max() if "year" in df.columns else np.nan,
        "n_years": df["year"].nunique() if "year" in df.columns else np.nan,
    }

    dataset_overview_rows.append(overview)

dataset_overview = pd.DataFrame(dataset_overview_rows)

print("Loaded datasets:")
display(list(datasets.keys()))

#display(dataset_overview)

Loaded datasets:


['isoc_r_dskl_i',
 'tgs00107',
 'demo_r_d2jan',
 'edat_lfse_04',
 'trng_lfse_04',
 'lfst_r_lfu3pers',
 'isoc_r_eb_ain2',
 'lfst_r_lfu3rt',
 'demo_r_pjanaggr3']

## Load processed datasets into separate dataframes

Each prepared dataset is loaded into its own dataframe.  
At this stage, I keep the datasets separate because they differ in regional level, time coverage, indicators, units, and structure. A combined feature table will only be created later after the relevant variables have been inspected and selected.

In [3]:
# -----------------------------
# Load each processed dataset into its own dataframe
# -----------------------------

digital_skills_df = pd.read_csv(PROCESSED_DIR / "digital_skills_nuts1_labeled_filtered.csv")
poverty_df = pd.read_csv(PROCESSED_DIR / "poverty_social_exclusion_nuts1_weighted.csv")
population_df = pd.read_csv(PROCESSED_DIR / "population_nuts2_total_long.csv")
education_df = pd.read_csv(PROCESSED_DIR / "education_attainment_labeled.csv")
lifelong_learning_df = pd.read_csv(PROCESSED_DIR / "lifelong_learning_labeled.csv")
unemployment_df = pd.read_csv(PROCESSED_DIR / "unemployment_education_region_labeled.csv")
enterprise_ai_df = pd.read_csv(PROCESSED_DIR / "enterprise_ai_adoption_labeled.csv")
nuts1_lookup = pd.read_csv(PROCESSED_DIR / "nuts1_lookup.csv")

# Store them in a dictionary as well for easier iteration
dataframes = {
    "digital_skills_df": digital_skills_df,
    "poverty_df": poverty_df,
    "population_df": population_df,
    "education_df": education_df,
    "lifelong_learning_df": lifelong_learning_df,
    "unemployment_df": unemployment_df,
    "enterprise_ai_df": enterprise_ai_df,
}

# -----------------------------
# Inspect structure of each dataframe
# -----------------------------

for name, df in dataframes.items():
    print("=" * 100)
    print(name)
    print("=" * 100)

    #print("\nDataFrame info:")
    #df.info()

    #print("\nFirst rows:")
    #display(df.head())

    print("\n")

digital_skills_df


poverty_df


population_df


education_df


lifelong_learning_df


unemployment_df


enterprise_ai_df




## Step 3: Select candidate variables

For each dataset, I inspect the available indicators, units, regional coverage, and missing values before making any selection decisions.  
The goal is to understand what is actually available in each dataset before deciding which variable to use for the index.

The inspection follows the order of the index pillars:

1. Digital skills — anchor dataset, defines the NUTS-1 scope
2. Poverty and social exclusion
3. Education attainment
4. Lifelong learning
5. Unemployment by education level
6. Enterprise AI adoption

### Digital skills — indicator and unit selection

The digital skills dataset is the anchor for the index. All 88 NUTS-1 regions have complete coverage (0 missing values).

**Decision:** `I_DSK2_BAB` + `PC_IND`

**Rationale:** `I_DSK2_BAB` (individuals with basic or above basic overall digital skills) summarises all five digital competence areas into a single threshold indicator. It offers the widest regional spread of all available indicators (19 %–85 %), making differences between regions clearly visible. The sub-dimension indicators (communication, content creation, information literacy, safety, problem solving) are already embedded in this composite score — using them separately would double-count the same information.

`PC_IND` (percentage of all individuals) is chosen over `PC_IND_IU3` (percentage of internet users only) because non-internet users are part of the AI literacy gap. They represent people who are not future-ready in the digital transformation and must not be excluded from the denominator.

**Alternatives considered:** `I_DSK2_AB` (above basic only, median 28 %) would set a stricter threshold but offers less regional differentiation. `PC_IND_IU3` would overstate skill levels by removing the digitally excluded population from the calculation.

In [4]:
# -----------------------------
# Step 3.1 — Digital skills pillar: filter to selected indicator and unit
# -----------------------------

DIGITAL_SKILLS_INDICATOR = "I_DSK2_BAB"
DIGITAL_SKILLS_UNIT = "PC_IND"

digital_skills_pillar = (
    digital_skills_df
    .loc[
        (digital_skills_df["indic_is"] == DIGITAL_SKILLS_INDICATOR) &
        (digital_skills_df["unit"] == DIGITAL_SKILLS_UNIT)
    ]
    [["geo", "nuts1_name", "country_code", "year", "value"]]
    .rename(columns={"value": "digital_skills_pct"})
    .reset_index(drop=True)
)

print(f"Indicator : {DIGITAL_SKILLS_INDICATOR}")
print(f"Unit      : {DIGITAL_SKILLS_UNIT}")
print(f"Regions   : {digital_skills_pillar['geo'].nunique()}")
print(f"Missing   : {digital_skills_pillar['digital_skills_pct'].isna().sum()}")
print(f"Range     : {digital_skills_pillar['digital_skills_pct'].min():.1f} % – {digital_skills_pillar['digital_skills_pct'].max():.1f} %")
print(f"Median    : {digital_skills_pillar['digital_skills_pct'].median():.1f} %")

display(digital_skills_pillar.sort_values("digital_skills_pct").head(10))

Indicator : I_DSK2_BAB
Unit      : PC_IND
Regions   : 88
Missing   : 0
Range     : 18.6 % – 85.2 %
Median    : 59.6 %


,geo,nuts1_name,country_code,year,digital_skills_pct
85,TRA,Kuzeydoğu Anadolu,TR,2025,18.56
87,TRC,Güneydoğu Anadolu,TR,2025,18.74
86,TRB,Ortadoğu Anadolu,TR,2025,19.43
82,TR7,Orta Anadolu,TR,2025,22.65
83,TR8,Batı Karadeniz,TR,2025,24.47
84,TR9,Doğu Karadeniz,TR,2025,25.25
81,TR6,Akdeniz,TR,2025,25.89
77,TR2,Batı Marmara,TR,2025,26.77
70,RO4,Macroregiunea Patru,RO,2025,27.09
67,RO1,Macroregiunea Unu,RO,2025,29.87


### Poverty and social exclusion — coverage assessment and reference year

The poverty dataset is already aggregated to NUTS-1 using population weights (prepared in `01_load_data.ipynb`). In the reference year 2025, two regions have incomplete NUTS-2 coverage:

| Region | Coverage | Missing NUTS-2 | Calculated value |
|---|---|---|---|
| FI1 — Manner-Suomi (Finland) | 75.0 % | FI19 Länsi-Suomi | 16.7 % |
| FRY — Régions Ultrapériphériques (France) | 85.7 % | FRY5 Mayotte | 41.4 % |

**Decision — FI1:** Value used as-is, no flag required. The three covered Finnish regions span only 2.4 percentage points (15.5 %–17.9 %), confirmed directly from our data. Länsi-Suomi is structurally comparable; estimated deviation from the true value is < 0.5 percentage points.

**Decision — FRY:** Value used with an explicit data caveat. Mayotte is the poorest French territory (documented in INSEE reports), but no EU-SILC-compliant poverty rate is available for it. External data are not added — mixing a non-EU-SILC source would break the methodological consistency of the index. The calculated value of 41.4 % likely understates the true NUTS-1 rate by a significant margin.

> **⚠ UI note for map visualisation:** FRY must be visually flagged in the final map (hatching, asterisk, or tooltip). Suggested text: *"Data covers 4 of 5 sub-regions. Mayotte is missing — actual poverty rate likely significantly higher (Mayotte is the poorest French territory; no EU-SILC-compliant rate available)."*

In [5]:
# -----------------------------
# Step 3.2 — Poverty pillar: filter to reference year 2025
# -----------------------------

POVERTY_REFERENCE_YEAR = 2025

# Flag regions with known data quality caveats
POVERTY_DATA_CAVEATS = {
    "FRY": "Mayotte (FRY5) missing — poverty rate likely understated by ~5 pp"
}

poverty_pillar = (
    poverty_df
    .loc[poverty_df["year"] == POVERTY_REFERENCE_YEAR]
    [["geo", "nuts1_name", "country_code", "year",
      "poverty_social_exclusion_rate", "population_coverage_share"]]
    .copy()
    .reset_index(drop=True)
)

poverty_pillar["data_caveat"] = poverty_pillar["geo"].map(POVERTY_DATA_CAVEATS).fillna("")

print(f"Reference year : {POVERTY_REFERENCE_YEAR}")
print(f"Regions        : {poverty_pillar['geo'].nunique()}")
print(f"Missing values : {poverty_pillar['poverty_social_exclusion_rate'].isna().sum()}")
print(f"Range          : {poverty_pillar['poverty_social_exclusion_rate'].min():.1f} % – {poverty_pillar['poverty_social_exclusion_rate'].max():.1f} %")
print(f"Median         : {poverty_pillar['poverty_social_exclusion_rate'].median():.1f} %")
print()
print("Regions with data caveats:")
display(poverty_pillar[poverty_pillar["data_caveat"] != ""][["geo", "nuts1_name", "poverty_social_exclusion_rate", "population_coverage_share", "data_caveat"]])

Reference year : 2025
Regions        : 89
Missing values : 0
Range          : 10.8 % – 41.4 %
Median         : 20.0 %

Regions with data caveats:


,geo,nuts1_name,poverty_social_exclusion_rate,population_coverage_share,data_caveat
51,FRY,RUP FR — Régions Ultrapériphériques Françaises,41.363619,0.857118,Mayotte (FRY5) missing — poverty rate likely u...


### Education attainment — indicator and age group selection

**Decision:** `ED0-2` + `Y25-64` (total sex, percentage)

**Rationale:** ED0-2 captures the share of the working-age population without any post-compulsory qualification — no degree beyond Hauptschule or Realschule. This is the most direct structural risk indicator for AI adaptability: people who never went beyond compulsory schooling have less practice in continuous learning and retraining. A selection effect reinforces this — motivated Realschule graduates typically continue to vocational training or further education and therefore appear in ED3-8, not ED0-2.

Y25-64 was selected because AI disruption affects all workers, not only younger cohorts. It is the only age group that covers the full working-age population and is the EU standard for education indicators. Y20-24 is distorted (many still in education), Y25-34 and Y30-34 are too narrow.

**Data limitation — age coverage:** Eurostat provides no age group beyond Y25-64 for this indicator. People aged 65 and over are not represented. This is a constraint of the source data, not an analytical choice. Digital skills and poverty contain no age breakdown at all (total population).

**Direction:** higher value = more people without post-compulsory qualification = larger AI literacy gap → must be inverted at index construction stage.

**Alternatives considered:** ED3-8 is mathematically the complement of ED0-2 (same signal, opposite direction). ED5-8 (tertiary only) would set too strict a threshold and exclude vocational training graduates.

In [6]:
# -----------------------------
# Step 3.3 — Education pillar: filter to selected indicator, age group, and reference year
# -----------------------------

nuts1_codes = set(nuts1_lookup["geo"])

EDUCATION_ISCED    = "ED0-2"
EDUCATION_AGE      = "Y25-64"
EDUCATION_SEX      = "T"
EDUCATION_UNIT     = "PC"
EDUCATION_YEAR     = 2025

education_pillar = (
    education_df
    .loc[
        (education_df["isced11"] == EDUCATION_ISCED) &
        (education_df["age"]     == EDUCATION_AGE) &
        (education_df["sex"]     == EDUCATION_SEX) &
        (education_df["unit"]    == EDUCATION_UNIT) &
        (education_df["year"]    == EDUCATION_YEAR) &
        (education_df["geo"].isin(nuts1_codes))
    ]
    [["geo", "geo_label", "year", "value"]]
    .rename(columns={"value": "low_education_pct", "geo_label": "nuts1_name"})
    .reset_index(drop=True)
)

print(f"ISCED level : {EDUCATION_ISCED}  (no post-compulsory qualification)")
print(f"Age group   : {EDUCATION_AGE}  (working-age population)")
print(f"Year        : {EDUCATION_YEAR}")
print(f"Regions     : {education_pillar['geo'].nunique()}")
print(f"Missing     : {education_pillar['low_education_pct'].isna().sum()}")
print(f"Range       : {education_pillar['low_education_pct'].min():.1f} % – {education_pillar['low_education_pct'].max():.1f} %")
print(f"Median      : {education_pillar['low_education_pct'].median():.1f} %")
print()
print("Note: higher value = larger gap — inversion required at index construction stage.")
print()
display(education_pillar.sort_values("low_education_pct", ascending=False).head(10))

ISCED level : ED0-2  (no post-compulsory qualification)
Age group   : Y25-64  (working-age population)
Year        : 2025
Regions     : 111
Missing     : 3
Range       : 3.4 % – 64.9 %
Median      : 16.4 %

Note: higher value = larger gap — inversion required at index construction stage.



,geo,nuts1_name,year,low_education_pct
110,TRC,Güneydoğu Anadolu,2025,64.9
109,TRB,Ortadoğu Anadolu,2025,60.9
108,TRA,Kuzeydoğu Anadolu,2025,59.7
86,PT2,Região Autónoma dos Açores,2025,56.7
106,TR8,Batı Karadeniz,2025,56.2
105,TR7,Orta Anadolu,2025,54.4
107,TR9,Doğu Karadeniz,2025,52.5
104,TR6,Akdeniz,2025,52.2
100,TR2,Batı Marmara,2025,51.6
101,TR3,Ege,2025,50.2


### Lifelong learning — indicator and age group selection

**Decision:** `PC` + `Y25-64` + `sex=T` (participation rate in education and training, last 4 weeks, age 25–64, total)

**Rationale:** Two age groups are available: `Y18-64` and `Y25-64`. Both have identical NUTS-1 coverage (109 / 123 in 2025) and are almost perfectly correlated (Pearson r = 0.986, Spearman r = 0.980), so the signal is equivalent. `Y25-64` is preferred for two reasons: (1) consistency with the Education pillar, which also uses Y25-64 as the working-age reference group; (2) conceptual clarity — 18–24-year-olds are often in initial education, making it harder to distinguish adult upskilling from initial training participation.

**Alternatives considered:** `Y18-64` — broader, matches the EU-LFS standard headline indicator. Rejected because the 18–24 age band introduces ambiguity between initial education and adult learning, and the correlation with Y25-64 is 0.986 (no informational gain).

**Reference year:** 2025 (consistent with all other pillars; 109 of 123 NUTS-1 regions have data).

**Missing regions:** 12 UK regions (systematic post-Brexit reporting gap — UK stopped submitting regional LFS data to Eurostat after 2024), IS0 (Iceland), ME0 (Montenegro).

**Value range (2025):** 2.6 % (MK0 — North Macedonia) to 40.3 % (SE1 — East Sweden), spread = 37.7 pp.

**Direction:** higher value = more lifelong learning participation = **lower** AI literacy gap risk → must be **inverted** during index construction (pillar score = 1 − normalised value).

**Data limitation — 4-week reference window:** The LFS-based indicator captures participation in any education or training activity in the **4 weeks prior to the survey interview**. This is a snapshot measure and systematically underrepresents people who learn intensively but infrequently — for example, someone who attends a week-long intensive course once a year would not be counted unless interviewed directly after it. A 12-month reference window (as used in the Adult Education Survey, AES) would be methodologically stronger for measuring lifelong learning behaviour. However, the AES is only available at **country level** (not NUTS-1) and is conducted approximately every five years (2007, 2011, 2016, 2022) — making it unsuitable for a regional annual index. `trng_lfse_04` is therefore the only viable option for NUTS-1 coverage. The 4-week indicator is the EU official DESI standard, ensuring cross-country comparability. This limitation should be noted in any published interpretation of the index.

In [7]:
# -----------------------------
# Step 3.4 — Lifelong learning pillar: filter to selected age group and reference year
# -----------------------------

LIFELONG_AGE  = "Y25-64"
LIFELONG_SEX  = "T"
LIFELONG_UNIT = "PC"
LIFELONG_YEAR = 2025

lifelong_learning_pillar = (
    lifelong_learning_df
    .loc[
        (lifelong_learning_df["age"]  == LIFELONG_AGE) &
        (lifelong_learning_df["sex"]  == LIFELONG_SEX) &
        (lifelong_learning_df["unit"] == LIFELONG_UNIT) &
        (lifelong_learning_df["year"] == LIFELONG_YEAR) &
        (lifelong_learning_df["geo"].str.len() == 3)
    ]
    [["geo", "geo_label", "year", "value"]]
    .rename(columns={"value": "lifelong_learning_pct", "geo_label": "nuts1_name"})
    .reset_index(drop=True)
)

print(f"Age group : {LIFELONG_AGE}  (working-age population, consistent with Education pillar)")
print(f"Year      : {LIFELONG_YEAR}")
print(f"Regions   : {lifelong_learning_pillar['geo'].nunique()}")
print(f"Missing   : {lifelong_learning_pillar['lifelong_learning_pct'].isna().sum()}")
print(f"Range     : {lifelong_learning_pillar['lifelong_learning_pct'].min():.1f} % – {lifelong_learning_pillar['lifelong_learning_pct'].max():.1f} %")
print(f"Median    : {lifelong_learning_pillar['lifelong_learning_pct'].median():.1f} %")
print()
print("Note: higher value = more participation = lower gap risk — inversion required at index construction stage.")
print()
display(lifelong_learning_pillar.sort_values("lifelong_learning_pct", ascending=False).head(10))

Age group : Y25-64  (working-age population, consistent with Education pillar)
Year      : 2025
Regions   : 123
Missing   : 14
Range     : 2.6 % – 40.3 %
Median    : 11.1 %

Note: higher value = more participation = lower gap risk — inversion required at index construction stage.



,geo,nuts1_name,year,lifelong_learning_pct
94,SE1,Östra Sverige,2025,40.3
95,SE2,Södra Sverige,2025,37.5
96,SE3,Norra Sverige,2025,34.5
27,DK0,Danmark,2025,31.0
40,FI1,Manner-Suomi,2025,28.1
41,FI2,Åland,2025,27.1
75,NL3,West-Nederland,2025,27.0
8,CH0,Schweiz/Suisse/Svizzera,2025,26.3
74,NL2,Oost-Nederland,2025,25.3
73,NL1,Noord-Nederland,2025,25.2


### Enterprise AI adoption — pillar excluded

**Decision:** `isoc_r_eb_ain2` (enterprise AI adoption by NUTS region) is **not included** in the index.

**Reasons:**

1. **Conceptual mismatch:** Enterprise AI adoption measures firm behaviour, not human AI literacy. The index measures where *people* lack the skills to adapt to AI-driven change — enterprise adoption is a demand-side context variable, not a gap indicator in itself.

2. **Structural data gap:** The dataset covers only 14 countries (AT, BE, BG, DK, ES, HR, HU, LT, NO, PL, RO, SE, SI, SK). Germany, France, Italy, and the Netherlands — the four largest EU economies — are entirely absent, not because their AI adoption is low, but because they do not report at NUTS-1 level. Including this pillar would reduce the index from 88 to 28 regions and systematically exclude major economies. A NaN in this context would be misread as "no AI adoption" when it actually means "no data reported."

**What is lost:** The demand-side exposure dimension — regions where enterprises adopt AI heavily create greater pressure on workers to adapt. This is documented as a deliberate scope decision, not an oversight.

In [8]:
# -----------------------------
# Load lfst_r_lfu3rt — unemployment rates by educational attainment
# Downloaded in 01_load_data.ipynb (see Additions section)
# Flags are embedded in value_raw (e.g. "15.3 u") — extracted here into a separate column
# -----------------------------

unemployment_rate_df = pd.read_csv(PROCESSED_DIR / "unemployment_rate_education_region_tidy.csv")

unemployment_rate_df["flag"] = (
    unemployment_rate_df["value_raw"]
    .astype(str)
    .str.extract(r"([a-zA-Z]+)\s*$")[0]
)

print(f"lfst_r_lfu3rt loaded from processed CSV.")
print(f"Shape: {unemployment_rate_df.shape}")
print(f"Years: {unemployment_rate_df['year'].min()} – {unemployment_rate_df['year'].max()}")
print(f"Unit:  {unemployment_rate_df['unit'].unique()}")
print(f"Flag breakdown: {unemployment_rate_df['flag'].value_counts().to_dict()}")

lfst_r_lfu3rt loaded from processed CSV.
Shape: (2008179, 10)
Years: 1999 – 2025
Unit:  <StringArray>
['PC']
Length: 1, dtype: str
Flag breakdown: {'u': 706428, 'bu': 223942, 'b': 144289, 'd': 18546, 'du': 6338, 'bd': 4800, 'bdu': 1533}


### Labour market vulnerability — dataset replacement and indicator selection

**Why lfst_r_lfu3rt instead of lfst_r_lfu3pers:**
`lfst_r_lfu3pers` contains unemployed persons in thousands (`THS_PER`) — absolute counts that make regions of different sizes incomparable (137k unemployed low-educated in NRW vs. 110k in all of Denmark says nothing without knowing the denominator). `lfst_r_lfu3rt` delivers the same breakdown as actual unemployment **rates** (`PC`), enabling direct cross-regional comparison.

**Decision:** `ED0-2` + `Y15-74` + `sex=T` (unemployment rate among people with less than upper secondary education, age 15–74, total)

**Rationale:** The unemployment rate among low-educated workers (ED0-2) directly measures whether a region structurally excludes its least-qualified workforce from the labour market. This is the cleanest available proxy for labour market vulnerability in the context of AI disruption: regions where low-educated people are already unemployed at high rates face compounded risk — those workers have both low adaptability (low education) and low labour market attachment.

`Y15-74` is selected because it offers the best NUTS-1 coverage (99 / 109 regions in 2025) and the lowest u-flag rate among all available age groups. Narrower age groups (e.g. Y25-34, Y55-64) have u-flag rates above 40 %, making them unsuitable. `Y20-64` would lose 5 additional regions compared to `Y15-74` with no methodological benefit.

**Age group note — intentional mismatch with Education and Lifelong Learning pillars:** Education (Y25-64) and Lifelong Learning (Y25-64) use the working-age educational attainment window — you need to wait until 25 to have completed formal education. Unemployment uses Y15-74, the ILO/Eurostat standard for labour market statistics, because labour market entry starts at 15 (end of compulsory schooling). Including 15–24 is intentional: early school leavers who are unemployed are a core vulnerability signal. These are different concepts with different appropriate age ranges; the mismatch is methodologically justified, not an error.

**Alternatives considered:** `lfst_r_lfu3pers` (original dataset) — rejected because absolute counts are not comparable across regions. Dropping the pillar entirely — considered but rejected because labour market exclusion adds a distinct dimension not covered by Education (structural qualification level) or Poverty (outcome): it measures the *transmission* between low qualification and economic exclusion.

**Reference year:** 2025 (109 NUTS-1 regions in dataset, 99 with values).

**Missing regions (10):** DE5 (Bremen), DE8 (Mecklenburg-Vorpommern), DEC (Saarland), FI2 (Åland), FRM (Corse), PL2/PL7/PL8/PL9 (Polish macro-regions), PT3 (Madeira). All are small regions with LFS sample sizes too small for reliable disaggregation by educational attainment — this gap is structural and present across **all** available age groups. No alternative age group or dataset resolves it. These regions will receive NaN for this pillar. UK absent from the dataset entirely (post-Brexit).

**Flag caveats:** 20 regions carry a `d`-flag (definition differs) — Spanish regions (ES3–ES7) and FRY. Values are used but noted. 12 further regions are `u`-flagged with values present — used with caveat.

**Value range (2025):** 4.1 % – 38.3 % (SK0 — Slovakia), median 11.9 %.

**Direction:** higher value = higher unemployment among low-educated = larger AI literacy gap risk → no inversion needed (higher = worse).

In [9]:
# -----------------------------
# Step 3.5 — Labour market vulnerability pillar: filter to selected indicator and reference year
# -----------------------------

UNEMPLOYMENT_ISCED = "ED0-2"
UNEMPLOYMENT_AGE   = "Y15-74"
UNEMPLOYMENT_SEX   = "T"
UNEMPLOYMENT_YEAR  = 2025

# Flags with values that should be noted but are used
UNEMPLOYMENT_FLAG_CAVEATS = {"d": "definition differs (Spain regions, FRY)", "u": "small sample — low reliability"}

unemployment_pillar = (
    unemployment_rate_df
    .loc[
        (unemployment_rate_df["isced11"] == UNEMPLOYMENT_ISCED) &
        (unemployment_rate_df["age"]     == UNEMPLOYMENT_AGE) &
        (unemployment_rate_df["sex"]     == UNEMPLOYMENT_SEX) &
        (unemployment_rate_df["year"]    == UNEMPLOYMENT_YEAR) &
        (unemployment_rate_df["geo"].str.len() == 3)
    ]
    [["geo", "year", "value", "flag"]]
    .rename(columns={"value": "unemployment_rate_low_edu_pct"})
    .reset_index(drop=True)
)

print(f"ISCED level : {UNEMPLOYMENT_ISCED}  (less than upper secondary)")
print(f"Age group   : {UNEMPLOYMENT_AGE}")
print(f"Year        : {UNEMPLOYMENT_YEAR}")
print(f"Regions     : {unemployment_pillar['geo'].nunique()}")
print(f"With value  : {unemployment_pillar['unemployment_rate_low_edu_pct'].notna().sum()}")
print(f"Missing     : {unemployment_pillar['unemployment_rate_low_edu_pct'].isna().sum()}")
print(f"Range       : {unemployment_pillar['unemployment_rate_low_edu_pct'].min():.1f} % – {unemployment_pillar['unemployment_rate_low_edu_pct'].max():.1f} %")
print(f"Median      : {unemployment_pillar['unemployment_rate_low_edu_pct'].median():.1f} %")
print()
print("Flag breakdown:")
print(unemployment_pillar["flag"].fillna("clean").value_counts().to_string())
print()
display(unemployment_pillar.dropna(subset=["unemployment_rate_low_edu_pct"]).sort_values("unemployment_rate_low_edu_pct", ascending=False).head(10))

ISCED level : ED0-2  (less than upper secondary)
Age group   : Y15-74
Year        : 2025
Regions     : 123
With value  : 99
Missing     : 24
Range       : 4.1 % – 38.3 %
Median      : 11.9 %

Flag breakdown:
flag
clean    81
u        22
d        20



,geo,year,unemployment_rate_low_edu_pct,flag
98,SK0,2025,38.3,NaN
94,SE1,2025,28.0,NaN
96,SE3,2025,27.5,NaN
95,SE2,2025,26.5,NaN
3,BE1,2025,25.4,NaN
55,FRY,2025,24.2,d
40,FI1,2025,23.1,NaN
38,ES6,2025,20.9,d
67,LT0,2025,19.6,NaN
89,RO2,2025,19.2,NaN


In [10]:
# -----------------------------
# Load demo_r_pjanaggr3 — population by broad age group
# Downloaded in 01_load_data.ipynb (see Additions section)
# -----------------------------

demographics_raw_df = pd.read_csv(PROCESSED_DIR / "demographics_age_structure_tidy.csv")

print(f"demo_r_pjanaggr3 loaded from processed CSV.")
print(f"Shape : {demographics_raw_df.shape}")
print(f"Years : {demographics_raw_df['year'].min()} – {demographics_raw_df['year'].max()}")
print(f"Ages  : {sorted(demographics_raw_df['age'].unique())}")

demo_r_pjanaggr3 loaded from processed CSV.
Shape : (1156680, 8)
Years : 1990 – 2025
Ages  : ['TOTAL', 'UNK', 'Y15-64', 'Y_GE65', 'Y_LT15']


### Demographics — indicator selection

**Decision:** Share of population aged 65 and over (`Y_GE65 / TOTAL × 100`), reference year 2025

**Rationale:** Regions with a higher share of elderly population tend to have structurally lower average digital literacy across all age groups, and face greater challenges in adapting to AI-driven transformation. The elderly share is the most direct and interpretable demographic proxy for this structural risk. It is calculated from absolute population counts provided by Eurostat (`demo_r_pjanaggr3`) — two values per region (Y_GE65, TOTAL) divided to produce a percentage.

**Why not the old-age dependency ratio (Y_GE65 / Y15-64):** The dependency ratio is the standard indicator in economic sustainability analyses (pension systems, care burden). For this index we are measuring structural AI literacy risk, not economic burden — the simpler and more directly interpretable share of 65+ is sufficient and avoids the additional complexity of a ratio between two age bands.

**Why not share of 55–64 (pre-retirement cohort):** This group is arguably the most directly at risk from AI displacement (still working, harder to retrain). However, it requires summing two separate 5-year age bands from a different dataset (`demo_r_pjangrp3`), and the information gain over total elderly share is marginal for a regional-level index.

**Dataset note:** `demo_r_pjanaggr3` provides broad age groups (`Y_LT15`, `Y15-64`, `Y_GE65`, `TOTAL`) directly at NUTS-1 level — no aggregation from NUTS-2 required. The existing `population_nuts2_total_long.csv` was not used here because it only contains total population counts (no age breakdown), having been prepared solely as a weighting table.

**Coverage:** 113 / 113 NUTS-1 regions with values in 2025 — best coverage of all pillars, no missing values.

**Value range (2025):** 5.5 % (TRC — Southeast Turkey) – 28.5 % (DEE — Saxony-Anhalt), median 21.3 %. Oldest regions: East Germany, Galicia (Spain), parts of Greece and France. Youngest: Turkish regions.

**Direction:** higher value = older population = larger AI literacy gap risk → no inversion needed (higher = worse).

In [11]:
# -----------------------------
# Step 3.6 — Demographics pillar: compute share of population aged 65+ at NUTS-1
# -----------------------------

DEMOGRAPHICS_YEAR = 2025

_nuts1_pop = (
    demographics_raw_df
    .loc[
        (demographics_raw_df["geo"].str.len() == 3) &
        (demographics_raw_df["sex"] == "T") &
        (demographics_raw_df["year"] == DEMOGRAPHICS_YEAR) &
        (demographics_raw_df["age"].isin(["TOTAL", "Y_GE65", "Y_LT15", "Y15-64"]))
    ]
    .pivot_table(index="geo", columns="age", values="value")
    .reset_index()
)
_nuts1_pop.columns.name = None

demographics_pillar = _nuts1_pop.rename(columns={
    "TOTAL":   "population_total",
    "Y_LT15":  "pop_lt15",
    "Y15-64":  "pop_15_64",
    "Y_GE65":  "pop_ge65",
}).copy()

demographics_pillar["share_ge65_pct"] = (
    demographics_pillar["pop_ge65"] / demographics_pillar["population_total"] * 100
).round(2)

print(f"Year    : {DEMOGRAPHICS_YEAR}")
print(f"Regions : {demographics_pillar['geo'].nunique()}")
print(f"Missing : {demographics_pillar['share_ge65_pct'].isna().sum()}")
print(f"Range   : {demographics_pillar['share_ge65_pct'].min():.1f} % – {demographics_pillar['share_ge65_pct'].max():.1f} %")
print(f"Median  : {demographics_pillar['share_ge65_pct'].median():.1f} %")
print()
display(demographics_pillar.sort_values("share_ge65_pct", ascending=False).head(10))

Year    : 2025
Regions : 113
Missing : 0
Range   : 5.5 % – 28.4 %
Median  : 21.3 %



,geo,population_total,pop_15_64,pop_ge65,pop_lt15,share_ge65_pct
25,DEE,2135597.0,1266266.0,607671.0,261660.0,28.45
19,DE8,1573597.0,935958.0,441327.0,196312.0,28.05
27,DEG,2100277.0,1247444.0,589004.0,263829.0,28.04
24,DED,4042422.0,2413440.0,1097767.0,531215.0,27.16
34,ES1,4323492.0,2703067.0,1163276.0,457149.0,26.91
15,DE4,2556747.0,1540998.0,677403.0,338346.0,26.49
33,EL6,2546748.0,1563401.0,660842.0,322505.0,25.95
51,FRI,6210839.0,3688735.0,1606543.0,915561.0,25.87
55,FRM,362253.0,216272.0,93364.0,52617.0,25.77
7,BG3,3107886.0,1882142.0,794912.0,430832.0,25.58


## Step 3.7: Index scope — EU27 definition and anchor extension

### Scope decision

The index is scoped to the **27 EU member states**. Non-EU countries with NUTS-1 data in Eurostat (Turkey, Norway, Switzerland, Serbia, Montenegro, North Macedonia, Albania, Iceland, UK) are excluded. This keeps the index within a common institutional, regulatory, and policy framework (EU AI Act, Digital Decade targets, Cohesion Policy) and enables actionable recommendations for EU decision-makers.

### Problem: 13 EU countries absent from digital skills anchor

The digital skills dataset (`isoc_r_dskl_i`) reports 13 EU member states at **country level** (2-char code, e.g. `DK`) rather than NUTS-1 level (3-char, e.g. `DK0`). The filter `geo.str.len() == 3` applied in `01_load_data.ipynb` excluded them. However, 12 of these 13 countries have exactly **one NUTS-1 region** — the country is the NUTS-1 region. The national value is therefore identical to the NUTS-1 value.

**Mapping applied:**

| 2-char | 3-char NUTS-1 | Country |
|---|---|---|
| CY | CY0 | Cyprus |
| CZ | CZ0 | Czechia |
| DK | DK0 | Denmark |
| EE | EE0 | Estonia |
| FI | FI1 | Finland ⚠ |
| HR | HR0 | Croatia |
| IE | IE0 | Ireland |
| LT | LT0 | Lithuania |
| LU | LU0 | Luxembourg |
| LV | LV0 | Latvia |
| MT | MT0 | Malta |
| SI | SI0 | Slovenia |
| SK | SK0 | Slovakia |

**⚠ Finland note:** Finland has two NUTS-1 regions — FI1 (Manner-Suomi, 5,605,317 inhabitants, 99.5 % of population) and FI2 (Åland, 30,654 inhabitants, 0.5 %). The national digital skills value is mapped to FI1 only. FI2 (Åland) is excluded from the index — its population share is negligible and it does not report separately in most Eurostat surveys.

### Problem: 5 EU countries absent from poverty dataset

`tgs00107` (the poverty dataset) only covers regions with NUTS-2 level EU-SILC data. Cyprus, Estonia, Luxembourg, Latvia, and Malta are not present at any geographic level in this dataset. As an alternative, `ilc_peps01n` provides the national AROPE rate from the same EU-SILC survey using the same definition. Since all five countries have a single NUTS-1 region, the national rate equals the NUTS-1 rate.

**Supplement source:** `ilc_peps01n` (Persons at risk of poverty or social exclusion, national level, Eurostat EU-SILC) — same methodology and reference year (2025) as `tgs00107`.

### Regions excluded despite EU membership

Seven EU NUTS-1 regions are excluded because they lack P5 (Unemployment rate for low-educated) due to insufficient LFS sample size. All belong to countries that remain represented by other NUTS-1 regions:

| Region | Country | Missing pillar |
|---|---|---|
| DE8 — Mecklenburg-Vorpommern | Germany | P5 Unemployment |
| FRM — Corse | France | P5 Unemployment |
| PL2, PL7, PL8, PL9 | Poland | P5 Unemployment |
| PT3 — Madeira | Portugal | P5 Unemployment |

In [12]:
# -----------------------------
# Step 3.7 — EU27 scope: extend anchor, supplement poverty, filter all pillars
# -----------------------------

EU27 = {
    "AT","BE","BG","CY","CZ","DE","DK","EE","EL","ES","FI","FR",
    "HR","HU","IE","IT","LT","LU","LV","MT","NL","PL","PT","RO","SE","SI","SK"
}

# Mapping: 2-char country code → 3-char NUTS-1 for single-region EU countries
# FI → FI1 (Manner-Suomi, 99.5 % of Finland's population; FI2/Åland excluded)
SINGLE_NUTS1_MAPPING = {
    "CY":"CY0","CZ":"CZ0","DK":"DK0","EE":"EE0","FI":"FI1",
    "HR":"HR0","IE":"IE0","LT":"LT0","LU":"LU0","LV":"LV0",
    "MT":"MT0","SI":"SI0","SK":"SK0"
}

# -----------------------------
# P1: extend digital skills with 13 mapped countries
# Raw TSV format: freq,indic_is,unit,geo\TIME_PERIOD (no sex dimension)
# -----------------------------
_dsk_rows = []
with open(RAW_DIR / "isoc_r_dskl_i.tsv") as _f:
    _header = _f.readline().strip().split("\t")
    _years  = [c.strip() for c in _header[1:]]
    for _line in _f:
        _parts = _line.strip().split("\t")
        _dims  = _parts[0].split(",")
        if len(_dims) < 4: continue
        _freq, _indic, _unit, _geo = _dims[0], _dims[1], _dims[2], _dims[3]
        if _geo not in SINGLE_NUTS1_MAPPING: continue
        if _indic != "I_DSK2_BAB" or _unit != "PC_IND": continue
        for _yr, _val in zip(_years, _parts[1:]):
            _m = re.search(r"([-+]?\d*\.?\d+)", str(_val))
            _dsk_rows.append({
                "geo": SINGLE_NUTS1_MAPPING[_geo],
                "year": int(_yr),
                "digital_skills_pct": float(_m.group(1)) if _m else None,
            })

_dsk_ext = pd.DataFrame(_dsk_rows)

digital_skills_pillar_eu = pd.concat([
    digital_skills_pillar,
    _dsk_ext[_dsk_ext["year"] == 2025][["geo","year","digital_skills_pct"]]
], ignore_index=True)

digital_skills_pillar_eu = digital_skills_pillar_eu[
    digital_skills_pillar_eu["geo"].str[:2].isin(EU27)
].drop_duplicates(subset=["geo"]).reset_index(drop=True)

print(f"P1 Digital Skills — EU27 Regionen: {digital_skills_pillar_eu['geo'].nunique()}")

# -----------------------------
# P2: supplement poverty for 5 missing EU countries via ilc_peps01n
# -----------------------------
_pov_rows = []
for _geo2, _nuts1 in [("CY","CY0"),("EE","EE0"),("LU","LU0"),("LV","LV0"),("MT","MT0")]:
    _url = (f"https://ec.europa.eu/eurostat/api/dissemination/statistics/1.0/data/"
            f"ilc_peps01n?format=JSON&geo={_geo2}&age=TOTAL&sex=T&unit=PC&lang=EN")
    _req = urllib.request.Request(_url, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(_req, timeout=10) as _r:
        _d = json.loads(_r.read())
    _t_idx = _d["dimension"]["time"]["category"]["index"]
    _vals  = _d.get("value", {})
    _pos   = _t_idx.get("2025")
    _val   = _vals.get(str(_pos)) if _pos is not None else None
    _pov_rows.append({
        "geo": _nuts1, "year": 2025,
        "poverty_social_exclusion_rate": _val,
        "nuts1_name": None,
        "data_caveat": "National rate from ilc_peps01n (country = single NUTS-1 region)",
        "population_coverage_share": 1.0,
    })

poverty_pillar_eu = pd.concat([
    poverty_pillar,
    pd.DataFrame(_pov_rows)
], ignore_index=True)

poverty_pillar_eu = poverty_pillar_eu[
    poverty_pillar_eu["geo"].str[:2].isin(EU27)
].drop_duplicates(subset=["geo"]).reset_index(drop=True)

print(f"P2 Poverty        — EU27 Regionen: {poverty_pillar_eu['geo'].nunique()}")

# -----------------------------
# P3–P6: filter to EU27
# -----------------------------
education_pillar_eu = education_pillar[
    education_pillar["geo"].str[:2].isin(EU27)].reset_index(drop=True)

lifelong_learning_pillar_eu = lifelong_learning_pillar[
    lifelong_learning_pillar["geo"].str[:2].isin(EU27)].reset_index(drop=True)

unemployment_pillar_eu = unemployment_pillar[
    unemployment_pillar["geo"].str[:2].isin(EU27)].reset_index(drop=True)

demographics_pillar_eu = demographics_pillar[
    demographics_pillar["geo"].str[:2].isin(EU27)].reset_index(drop=True)

print(f"P3 Education      — EU27 Regionen: {education_pillar_eu['geo'].nunique()}")
print(f"P4 Lifelong       — EU27 Regionen: {lifelong_learning_pillar_eu['geo'].nunique()}")
print(f"P5 Unemployment   — EU27 Regionen: {unemployment_pillar_eu[unemployment_pillar_eu['unemployment_rate_low_edu_pct'].notna()]['geo'].nunique()}")
print(f"P6 Demographics   — EU27 Regionen: {demographics_pillar_eu['geo'].nunique()}")

# -----------------------------
# Coverage summary across all pillars
# -----------------------------
_p1s = set(digital_skills_pillar_eu["geo"])
_p2s = set(poverty_pillar_eu["geo"])
_p3s = set(education_pillar_eu["geo"])
_p4s = set(lifelong_learning_pillar_eu["geo"])
_p5s = set(unemployment_pillar_eu[unemployment_pillar_eu["unemployment_rate_low_edu_pct"].notna()]["geo"])
_p6s = set(demographics_pillar_eu["geo"])

_all_geos = sorted(_p1s | _p2s | _p3s | _p4s | _p5s | _p6s)
_coverage = pd.DataFrame({
    "geo":         _all_geos,
    "P1_Digital":  ["✓" if g in _p1s else "✗" for g in _all_geos],
    "P2_Poverty":  ["✓" if g in _p2s else "✗" for g in _all_geos],
    "P3_Education":["✓" if g in _p3s else "✗" for g in _all_geos],
    "P4_Lifelong": ["✓" if g in _p4s else "✗" for g in _all_geos],
    "P5_Unemploy": ["✓" if g in _p5s else "✗" for g in _all_geos],
    "P6_Demograph":["✓" if g in _p6s else "✗" for g in _all_geos],
})
_coverage["n_pillars"] = _coverage.iloc[:, 1:].apply(lambda r: (r=="✓").sum(), axis=1)

print(f"\nEU27-Regionen gesamt:         {len(_coverage)}")
print(f"Mit allen 6 Pillars:          {(_coverage['n_pillars']==6).sum()}")
print(f"Mit 5 Pillars (1 fehlt):      {(_coverage['n_pillars']==5).sum()}")
print(f"\nUnvollständige Regionen (werden im Merge-Schritt ausgeschlossen):")
display(_coverage[_coverage["n_pillars"] < 6].sort_values("n_pillars"))

P1 Digital Skills — EU27 Regionen: 87


P2 Poverty        — EU27 Regionen: 91
P3 Education      — EU27 Regionen: 92
P4 Lifelong       — EU27 Regionen: 92
P5 Unemployment   — EU27 Regionen: 82
P6 Demographics   — EU27 Regionen: 92

EU27-Regionen gesamt:         92
Mit allen 6 Pillars:          80
Mit 5 Pillars (1 fehlt):      9

Unvollständige Regionen (werden im Merge-Schritt ausgeschlossen):


,geo,P1_Digital,P2_Poverty,P3_Education,P4_Lifelong,P5_Unemploy,P6_Demograph,n_pillars
40,FI2,✗,✗,✓,✓,✗,✓,3
14,DE5,✗,✓,✓,✓,✗,✓,4
21,DEC,✗,✓,✓,✓,✗,✓,4
17,DE8,✓,✓,✓,✓,✗,✓,5
20,DEB,✗,✓,✓,✓,✓,✓,5
25,DEG,✗,✓,✓,✓,✓,✓,5
53,FRM,✓,✓,✓,✓,✗,✓,5
73,PL2,✓,✓,✓,✓,✗,✓,5
77,PL7,✓,✓,✓,✓,✗,✓,5
78,PL8,✓,✓,✓,✓,✗,✓,5


## Step 4: Merge to final feature table

All six pillar DataFrames are joined into a single wide table — one row per NUTS-1 region, one column per pillar.

**Decision:** Inner join on `geo` across all six pillars, keeping the P1 Digital Skills EU27 scope as the binding constraint.

**Rationale:** An inner join retains only regions that appear in all six DataFrames. P1 (87 regions) is the narrowest dataset and defines the index scope. P2–P6 all have ≥ 87 EU27 regions, so no region is lost in the join. The seven regions with NaN in P5 (DE8, FRM, PL2, PL7, PL8, PL9, PT3) survive the join because their rows exist in `unemployment_pillar_eu` — the key `geo` is present, only the value is missing. Inner join checks key presence, not value completeness.

**Alternatives considered:** Left join on P1 would yield the same result here (P1 is the binding constraint). Full outer join was rejected — it would introduce non-EU regions present in P2–P6 but absent from the Digital Skills anchor.

**Result:** 87 regions × 7 columns (geo + 6 pillar values). 80 regions fully complete; 7 with NaN in P5 (accepted per DECISION.md §8).

In [13]:
# -----------------------------
# Step 4 — Merge all 6 pillar DataFrames to final feature table
# -----------------------------

p1 = digital_skills_pillar_eu[["geo", "digital_skills_pct"]]
p2 = poverty_pillar_eu[["geo", "poverty_social_exclusion_rate"]]
p3 = education_pillar_eu[["geo", "low_education_pct"]]
p4 = lifelong_learning_pillar_eu[["geo", "lifelong_learning_pct"]]
p5 = unemployment_pillar_eu[["geo", "unemployment_rate_low_edu_pct", "flag"]]
p6 = demographics_pillar_eu[["geo", "share_ge65_pct"]]

feature_table = (
    p1
    .merge(p2, on="geo", how="inner")
    .merge(p3, on="geo", how="inner")
    .merge(p4, on="geo", how="inner")
    .merge(p5, on="geo", how="inner")
    .merge(p6, on="geo", how="inner")
    .sort_values("geo")
    .reset_index(drop=True)
    .rename(columns={"flag": "unemployment_flag"})
)

# -----------------------------
# Save feature table
# -----------------------------

feature_table.to_csv(PROCESSED_DIR / "pillar_feature_table.csv", index=False)

# -----------------------------
# Save excluded regions as a programmatic artifact
# Regions present in P1 anchor but missing P5 (insufficient LFS sample size)
# -----------------------------

excluded_regions = (
    feature_table[feature_table["unemployment_rate_low_edu_pct"].isna()][["geo"]]
    .assign(
        missing_pillar="P5 — Unemployment rate (low-educated)",
        reason="LFS sample size too small for ED0-2 disaggregation at NUTS-1 level",
        included_in_index=False
    )
    .reset_index(drop=True)
)

excluded_regions.to_csv(PROCESSED_DIR / "excluded_regions.csv", index=False)

# -----------------------------
# Register feature table in project data catalog
# -----------------------------

_catalog = pd.read_csv(PROCESSED_DIR / "project_data_catalog.csv")

if "pillar_feature_table" not in _catalog["dataset_code"].values:
    _ft_entry = {
        "dataset_code":   "pillar_feature_table",
        "dataset_name":   "AI Literacy Gap Index — final pillar feature table",
        "pillar":         "All pillars",
        "processed_file": "pillar_feature_table.csv",
        "main_use":       "Input for index construction (03_index_construction.ipynb)",
        "file_exists":    True,
        "n_rows":         feature_table.shape[0],
        "n_columns":      feature_table.shape[1],
        "min_year":       2025,
        "max_year":       2025,
        "n_years":        1,
        "downloaded_at":  _catalog["downloaded_at"].iloc[0] if "downloaded_at" in _catalog.columns else None,
    }
    _catalog = pd.concat([_catalog, pd.DataFrame([_ft_entry])], ignore_index=True)
    _catalog.to_csv(PROCESSED_DIR / "project_data_catalog.csv", index=False)

# -----------------------------
# Summary
# -----------------------------

print(f"Shape                    : {feature_table.shape}")
print(f"Regions total            : {len(feature_table)}")
print(f"Fully complete (0 NaN)   : {feature_table.dropna().shape[0]}")
print(f"NaN in P5 (unemployment) : {feature_table['unemployment_rate_low_edu_pct'].isna().sum()}")
print(f"Other NaN                : {feature_table.drop(columns=['unemployment_rate_low_edu_pct','unemployment_flag']).isna().sum().sum()}")
print()
print("Excluded regions (NaN in P5):")
print(excluded_regions[["geo", "reason"]].to_string(index=False))
print()
print(f"Catalog entries          : {len(_catalog)}")
print()
display(feature_table.describe().round(2))

Shape                    : (87, 8)
Regions total            : 87
Fully complete (0 NaN)   : 30
NaN in P5 (unemployment) : 7
Other NaN                : 0

Excluded regions (NaN in P5):
geo                                                             reason
DE8 LFS sample size too small for ED0-2 disaggregation at NUTS-1 level
FRM LFS sample size too small for ED0-2 disaggregation at NUTS-1 level
PL2 LFS sample size too small for ED0-2 disaggregation at NUTS-1 level
PL7 LFS sample size too small for ED0-2 disaggregation at NUTS-1 level
PL8 LFS sample size too small for ED0-2 disaggregation at NUTS-1 level
PL9 LFS sample size too small for ED0-2 disaggregation at NUTS-1 level
PT3 LFS sample size too small for ED0-2 disaggregation at NUTS-1 level

Catalog entries          : 10



,digital_skills_pct,poverty_social_exclusion_rate,low_education_pct,lifelong_learning_pct,unemployment_rate_low_edu_pct,share_ge65_pct
count,87.00,87.00,87.00,87.00,80.00,87.00
mean,59.62,21.18,17.59,13.99,13.36,21.83
std,12.32,6.49,10.86,7.27,6.16,2.95
min,27.09,10.81,3.40,3.30,4.10,12.99
25%,51.53,16.92,10.40,9.00,9.38,20.02
50%,62.00,19.91,15.00,12.80,12.65,21.66
75%,67.26,24.17,20.90,16.80,16.12,23.88
max,85.17,41.36,56.70,40.30,38.30,28.45
